# 16 — Evaluation-Driven Prompt Optimization

## Scenario
We have a system that extracts `Company Name` and `Sentiment` from news headlines. 
We discover a failure in production: our prompt is failing to extract the company name when the headline uses a ticker symbol instead of the full name.

**The Problem:** We need to update our prompt to fix the edge case, but we must ensure we don't cause a *global regression* (breaking the standard queries that were previously working).

In [ ]:
import os
from enum import Enum
from pydantic import BaseModel, Field
from google import genai
from google.genai import types

client = genai.Client()
MODEL_ID = 'gemini-2.5-flash'

# 1. Define the Schema
class SentimentEnum(str, Enum):
    POSITIVE = "Positive"
    NEGATIVE = "Negative"
    NEUTRAL = "Neutral"

class ExtractionResult(BaseModel):
    company_name: str
    sentiment: SentimentEnum

# 2. Define the Evaluation Dataset
dataset = [
    # Standard Cases (Working)
    {"headline": "Northstar posts record profits in Q3", "expected_company": "Northstar", "expected_sentiment": "Positive"},
    {"headline": "Acme Corp faces massive lawsuit over safety violations", "expected_company": "Acme Corp", "expected_sentiment": "Negative"},
    
    # Edge Case (Failing in production)
    {"headline": "$GOOG rallies 5% on AI announcements", "expected_company": "Google", "expected_sentiment": "Positive"},
]


## Step 1: Evaluating the Baseline

We run our evaluation loop on the current production prompt to confirm the failure.

In [ ]:
baseline_prompt_template = """\nExtract the company name and sentiment from the following headline.\nHeadline: {headline}\n"""

def evaluate_prompt(prompt_template, dataset):
    correct_company = 0
    correct_sentiment = 0
    total = len(dataset)
    
    for i, data in enumerate(dataset):
        headline = data["headline"]
        expected_company = data["expected_company"]
        expected_sentiment = data["expected_sentiment"]
        
        prompt = prompt_template.format(headline=headline)
        
        response = client.models.generate_content(
            model=MODEL_ID,
            contents=prompt,
            config=types.GenerateContentConfig(
                temperature=0.0,
                response_mime_type="application/json",
                response_schema=ExtractionResult,
            )
        )
        
        result = ExtractionResult.model_validate_json(response.text)
        
        comp_match = (result.company_name == expected_company)
        sent_match = (result.sentiment.value == expected_sentiment)
        
        if comp_match: correct_company += 1
        if sent_match: correct_sentiment += 1
            
        print(f"Headline {i+1}:")
        print(f"  Company:   [{'PASS' if comp_match else 'FAIL'}] Expected: '{expected_company}', Got: '{result.company_name}'")
        print(f"  Sentiment: [{'PASS' if sent_match else 'FAIL'}] Expected: '{expected_sentiment}', Got: '{result.sentiment.value}'")
        
    print(f"\nTotal Company Accuracy:   {(correct_company/total)*100:.1f}%")
    print(f"Total Sentiment Accuracy: {(correct_sentiment/total)*100:.1f}%")

print("--- BASELINE EVALUATION ---")
evaluate_prompt(baseline_prompt_template, dataset)


## Step 2: The Naive Fix (Causing a Regression)

We see the baseline failed to extract "Google" from "$GOOG". 
We write a naive fix: "If you see a ticker symbol, use the company name."

In [ ]:
naive_fix_template = """\nExtract the company name and sentiment from the following headline.\nIf you see a ticker symbol, replace it with the full company name.\nOtherwise, ONLY output the exact words found in the headline.\nHeadline: {headline}\n"""

print("\n--- NAIVE FIX EVALUATION ---")
evaluate_prompt(naive_fix_template, dataset)

# Notice what happened: We fixed the edge case (Headline 3 now passes).
# BUT we broke Headline 2! By adding the instruction "ONLY output the exact words", 
# the model might get confused or fail to extract the right entity if it doesn't match perfectly.
# This is a GLOBAL REGRESSION.

## Conclusion: Evaluation-Driven Optimization

If we didn't have an automated evaluation loop running against our *entire* dataset, we would have deployed the naive fix, thinking we solved the problem, when in reality we just broke a different part of the system.

Prompt engineering is not about "fixing the prompt until the test case works." It is about writing a hypothesis, changing one variable, and running a regression test across hundreds of examples.